In [ ]:
from langsmith import Client, evaluate, traceable
#from inventory_agent import run
from langchain.tools import tool
from utils import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

D:\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@traceable
def target(input: dict) -> dict:
    question = inputs["question"]
    amswer = run(question)
    return {"answer": amswer}

client = Client()

dataset_name = "inventorydata"
if not client.has_dataset(dataset_name=dataset_name):
    client.create_dataset(dataset_name=dataset_name)

    client.create_example(
        dataset_name = dataset_name,
        examples =[
            {
                "inputs":{"question": "what is the stock status of iPhone 15?"}
                "outputs":{"answer": "the iPhone 15 is currently in stock with 2 units available"}
            },
            {
                "inputs":{"question": "Is AirPods pro available?"}
                "outputs":{"answer": "the AirPods pro is currently out of stock. there are 0 available"}
            },
            {
                "inputs":{"question": "How many iPhone 15 units are available?"}
                "outputs":{"answer": "the iPhone 15 is currently in stock with 2 units available"}
            },
            {
                "inputs":{"question": "Do you have Samsung Galaxy S23?"}
                "outputs":{"answer": "the product is not available in our inventory"}
            },
            {
                "inputs":{"question": "Can you tell me the recipe of Vada Pav?"}
                "outputs":{"answer": "Sorry i can not assist with that"}
            }
        ]
    )

def semantic_match(example, run):
    expected = example.outputs["answer"]
    actual = run.outputs["answer"]
    sim = cosine_similarity(expected, actual)
    return{
        "key":"semantic_match",
        "score": float(sim)
    }

evaluate(
    target,
    client= client,
    data = dataset_name,
    evaluators=[semantic_match],
    experiment_prefix="inventory_agent_evaluation_qwen3-32b"
    
)


    

